# SIMBES — Módulo 4: Eléctrico y VSD

**Objetivo:** Entender los fenómenos eléctricos críticos en un sistema BES:
1. Caída de voltaje en el cable (corrección por temperatura)
2. Regla de Arrhenius — degradación de aislamiento
3. THD por topología VSD — norma IEEE 519-2014
4. Selección de materiales — NACE MR0175 / ISO 15156

---
**Fuentes:** IEEE 519-2014 · NACE MR0175/ISO 15156 · Arrhenius (1889) · ABB TN060  
**Unidades:** sistema mixto campo (ft, AWG, A, V, °C)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..', 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from physics.electrical import (
    cable_resistance_corrected,
    cable_voltage_drop,
    arrhenius_life_factor,
    thd_estimate,
    material_recommendation,
    CABLE_RESISTANCE_OHM_PER_1000FT,
    VSD_THD_TYPICAL,
    IEEE_519_LIMIT_PCT,
)

plt.rcParams.update({
    'figure.facecolor': '#0B0F1A',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1E293B',
    'axes.labelcolor':  '#CBD5E1',
    'xtick.color':      '#64748B',
    'ytick.color':      '#64748B',
    'text.color':       '#CBD5E1',
    'grid.color':       '#1E293B',
    'grid.linestyle':   '--',
    'legend.facecolor': '#111827',
    'legend.edgecolor': '#1E293B',
    'font.family':      'monospace',
})
print('Imports OK ✓')

## 1. Cable Eléctrico — Caída de Voltaje

### Física

La resistencia del conductor de cobre varía con la temperatura:

$$
R_T = R_{20} \times (1 + \alpha \cdot (T_{\text{avg}} - 20)) \quad [\Omega]
$$

Donde $\alpha_{\text{Cu}} = 0.00393\ /°C$ y $T_{\text{avg}}$ es la temperatura promedio del cable.

Caída de voltaje en cable trifásico:

$$
V_{\text{drop}} = I \times R_T \times 3 \quad [V]
$$

**Límites operativos:** < 5% = OK · 5–10% = Advertencia · > 10% = Peligro

In [ ]:
# Parámetros base — Pozo Flamingo-4
depth_ft    = 6500
I_amps      = 95
T_bottom_C  = 175
T_surface_C = 25

AWG_OPTIONS = [1, 2, 4, 6, 8, 10, 12, 14]

print(f"Pozo Flamingo-4: {depth_ft} ft · I={I_amps} A · T_fondo={T_bottom_C}°C")
print("-" * 60)
print(f"{'AWG':>5} {'R_base[Ω]':>12} {'R_corr[Ω]':>12} {'V_drop[V]':>12} {'%drop':>8} {'Estado':>12}")
print("-" * 60)

for awg in AWG_OPTIONS:
    r = cable_voltage_drop(awg, depth_ft, I_amps, T_bottom_C, T_surface_C)
    rc = cable_resistance_corrected(awg, depth_ft, T_bottom_C, T_surface_C)
    estado = '❌ PELIGRO' if r['pct_drop'] > 10 else ('⚠️  AVISO' if r['warning_5pct'] else '✅ OK')
    print(f"  #{awg:>2}  {rc['R_base_ohm']:>12.4f} {r['R_corrected_ohm']:>12.4f} {r['V_drop_V']:>12.1f} {r['pct_drop']:>8.1f}  {estado}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: sensibilidad a AWG
ax = axes[0]
pcts  = [cable_voltage_drop(awg, depth_ft, I_amps, T_bottom_C)['pct_drop'] for awg in AWG_OPTIONS]
bars  = ax.bar([f'#{a}' for a in AWG_OPTIONS], pcts, color=[
    '#EF4444' if p > 10 else '#F59E0B' if p > 5 else '#22C55E' for p in pcts
], edgecolor='#1E293B')
ax.axhline(5,  color='#F59E0B', lw=1.5, ls='--', label='Advertencia (5%)')
ax.axhline(10, color='#EF4444', lw=1.5, ls='--', label='Peligro (10%)')
ax.set_xlabel('Calibre AWG')
ax.set_ylabel('Caída de voltaje [%]')
ax.set_title(f'Sensibilidad V_drop por calibre AWG\n{depth_ft} ft · {I_amps} A · T={T_bottom_C}°C', pad=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)

# Panel derecho: sensibilidad a profundidad
ax2 = axes[1]
depths = np.linspace(1000, 14000, 100)
for awg, color in [(4, '#22C55E'), (6, '#F472B6'), (8, '#F59E0B'), (10, '#EF4444')]:
    pcts_d = [cable_voltage_drop(awg, d, I_amps, T_bottom_C)['pct_drop'] for d in depths]
    ax2.plot(depths, pcts_d, color=color, lw=2, label=f'AWG #{awg}')

ax2.axhline(5,  color='#F59E0B', lw=1.2, ls='--')
ax2.axhline(10, color='#EF4444', lw=1.2, ls='--')
ax2.axhspan(5,  10, alpha=0.06, color='#F59E0B')
ax2.axhspan(10, 35, alpha=0.06, color='#EF4444')
ax2.set_xlabel('Profundidad [ft]')
ax2.set_ylabel('Caída de voltaje [%]')
ax2.set_title(f'V_drop vs Profundidad por AWG\nI={I_amps} A · T={T_bottom_C}°C', pad=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.4)
ax2.set_xlim(1000, 14000)

plt.suptitle('Cable BES — Sensibilidad de Caída de Voltaje', color='#CBD5E1', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 2. Regla de Arrhenius — Degradación del Aislamiento

$$
\frac{\tau_2}{\tau_1} = 2^{(T_1 - T_2)/10}
$$

**Interpretación:** Por cada **10°C** sobre el límite nominal, la vida útil del aislamiento se **reduce a la mitad**.

| Clase aislamiento | T° límite | Aplicación típica BES |
|-------------------|-----------|----------------------|
| Clase F | 155°C | Pozos poco profundos |
| Clase H | 180°C | BES estándar |
| Clase C | 220°C | Pozos de alta temperatura |

In [ ]:
INSULATION_CLASSES = [
    ('Clase F', 155, '#38BDF8'),
    ('Clase H', 180, '#F472B6'),
    ('Clase C', 220, '#22C55E'),
]

# Escenarios del Pozo Flamingo-4
print("Regla de Arrhenius — Pozo Flamingo-4")
print(f"T_operación = {T_bottom_C}°C")
print("-" * 55)
for name, T_rated, _ in INSULATION_CLASSES:
    r = arrhenius_life_factor(T_bottom_C, T_rated)
    print(f"  {name:<10} (límite {T_rated}°C) → {r['message']}")

# Escenario con temperatura elevada
print("\nEscalada de temperatura para aislamiento Clase H (180°C):")
print("-" * 50)
for T_op in [170, 180, 190, 200, 210, 220, 230]:
    r = arrhenius_life_factor(T_op, 180)
    bar = '█' * int(r['pct_life_remaining'] / 5)
    print(f"  T={T_op}°C  vida={r['pct_life_remaining']:6.1f}%  {bar}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: curvas de Arrhenius por clase de aislamiento
ax = axes[0]
T_range = np.linspace(100, 250, 300)

for name, T_rated, color in INSULATION_CLASSES:
    life_pct = [
        arrhenius_life_factor(T, T_rated)['pct_life_remaining']
        for T in T_range
    ]
    ax.plot(T_range, life_pct, color=color, lw=2, label=f'{name} ({T_rated}°C)')
    ax.axvline(T_rated, color=color, lw=0.8, ls=':', alpha=0.6)

ax.axvline(T_bottom_C, color='white', lw=1.5, ls='--', alpha=0.7,
           label=f'T_op actual ({T_bottom_C}°C)')
ax.axhline(50, color='#F59E0B', lw=1, ls='--', alpha=0.6, label='50% vida útil')
ax.set_xlabel('Temperatura de Operación [°C]')
ax.set_ylabel('Vida útil relativa [%]')
ax.set_title('Curvas de Arrhenius — Vida del Aislamiento', pad=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_xlim(100, 250)
ax.set_ylim(0, 105)

# Panel derecho: semivida por exceso de temperatura (ΔT)
ax2 = axes[1]
dT_range = np.linspace(0, 70, 200)
life_curve = [2**(-dT/10) * 100 for dT in dT_range]

ax2.fill_between(dT_range, life_curve, alpha=0.15, color='#F472B6')
ax2.plot(dT_range, life_curve, color='#F472B6', lw=2.5)

# Marcar las semividas
for halvings in [1, 2, 3, 4]:
    dT = halvings * 10
    lf = 2**(-halvings) * 100
    ax2.scatter([dT], [lf], color='white', zorder=5, s=40)
    ax2.annotate(f'{lf:.1f}%', xy=(dT, lf), xytext=(dT + 1, lf + 3),
                 fontsize=8, color='#CBD5E1')

ax2.set_xlabel('Exceso de temperatura ΔT [°C]')
ax2.set_ylabel('Vida útil relativa [%]')
ax2.set_title('Curva τ/τ₀ = 2^(−ΔT/10)\n"Cada 10°C → vida ÷ 2"', pad=10)
ax2.grid(True, alpha=0.4)
ax2.set_xlim(0, 70)
ax2.set_ylim(0, 105)

plt.suptitle('Regla de Arrhenius — Degradación del Aislamiento', color='#CBD5E1', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. THD — Distorsión Armónica Total por Topología VSD

### Norma IEEE 519-2014

Límite de THD en voltaje en el Punto de Acoplamiento Común (PCC): **THDv < 5%**

| Topología | THD típico | Cumple IEEE 519 |
|-----------|-----------|------------------|
| 6 pulsos estándar | ~30% | ❌ |
| 12 pulsos | ~17.5% | ❌ |
| 18 pulsos | ~4% | ✅ |
| AFE (Active Front End) | ~2.5% | ✅ |
| Filtro Activo | ~1.5% | ✅ |

In [ ]:
print("THD por topología VSD — IEEE 519-2014 (límite 5%)")
print("-" * 60)
for topo_key in VSD_THD_TYPICAL:
    r = thd_estimate(topo_key)
    cumple = '✅ CUMPLE' if r['complies_ieee519'] else '❌ NO CUMPLE'
    print(f"  {r['topology_desc']:<35} THD={r['THD_pct']:5.1f}%  {cumple}")
    if r['recommendation']:
        print(f"    → {r['recommendation']}")

In [ ]:
topos    = list(VSD_THD_TYPICAL.keys())
labels   = [VSD_THD_TYPICAL[t]['desc'].replace(' ', '\n') for t in topos]
thd_vals = [VSD_THD_TYPICAL[t]['THD_pct'] for t in topos]
colors   = ['#EF4444' if v >= IEEE_519_LIMIT_PCT else '#22C55E' for v in thd_vals]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, thd_vals, color=colors, edgecolor='#1E293B', width=0.6)

ax.axhline(IEEE_519_LIMIT_PCT, color='#22C55E', lw=2, ls='--',
           label=f'IEEE 519-2014 límite ({IEEE_519_LIMIT_PCT}%)')
ax.axhspan(0, IEEE_519_LIMIT_PCT, alpha=0.07, color='#22C55E')

for bar, val in zip(bars, thd_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.4, f'{val}%',
            ha='center', va='bottom', fontsize=10, color='#CBD5E1',
            fontfamily='monospace')

ok_patch  = mpatches.Patch(color='#22C55E', label='Cumple IEEE 519')
nok_patch = mpatches.Patch(color='#EF4444', label='No cumple IEEE 519')
ax.legend(handles=[ok_patch, nok_patch, ax.get_lines()[0]], fontsize=9)

ax.set_ylabel('THD [%]')
ax.set_title('THD por Topología VSD — Comparación IEEE 519-2014', pad=12)
ax.set_ylim(0, 35)
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Selección de Materiales — NACE MR0175 / ISO 15156

La norma aplica cuando existe H₂S en el fluido producido. El H₂S causa **Sulfide Stress Cracking (SSC)** en aceros de alta resistencia y degrada los elastómeros comunes.

In [ ]:
casos = [
    {'T_C': 120, 'H2S': False, 'solvent': False, 'desc': 'Estándar (T<140°C, sin H₂S)'},
    {'T_C': 160, 'H2S': False, 'solvent': False, 'desc': 'Alta temperatura (T>140°C, sin H₂S)'},
    {'T_C': 130, 'H2S': True,  'solvent': False, 'desc': 'Gas amargo (H₂S presente)'},
    {'T_C': 160, 'H2S': True,  'solvent': False, 'desc': 'Alta T° + H₂S (crítico)'},
    {'T_C': 140, 'H2S': False, 'solvent': True,  'desc': 'Inyección de solventes'},
]

print("NACE MR0175 / ISO 15156 — Selección de materiales")
print("=" * 70)
for c in casos:
    r = material_recommendation(c['T_C'], c['H2S'], c['solvent'])
    print(f"\n  Caso: {c['desc']}")
    print(f"    Elastómero  : {r['elastomer_recommendation']}")
    print(f"    Cable       : {r['cable_jacket']}")
    print(f"    Normas      : {', '.join(r['applicable_standards'])}")
    if r['warnings']:
        for w in r['warnings']:
            print(f"    ⚠️  {w}")
    else:
        print(f"    ✅ Materiales estándar adecuados.")

## 5. Análisis Integrado — Caso Pozo Flamingo-4

Aplicación completa de los 4 subsistemas eléctricos al caso del pozo Flamingo-4.

In [ ]:
# ── Escenario final (post-optimización) ─────────────────────────────
AWG_FINAL     = 4
DEPTH_FT      = 6500
I_AMPS        = 95
T_BOTTOM_C    = 175
T_RATED_C     = 180   # Clase H
VSD_TOPO      = '18pulse'
H2S_PRESENT   = True

cable = cable_voltage_drop(AWG_FINAL, DEPTH_FT, I_AMPS, T_BOTTOM_C)
arrh  = arrhenius_life_factor(T_BOTTOM_C, T_RATED_C)
thd   = thd_estimate(VSD_TOPO)
nace  = material_recommendation(T_BOTTOM_C, H2S_PRESENT)

print("POZO FLAMINGO-4 — Resumen eléctrico post-optimización")
print("=" * 55)
print(f"  Cable AWG #{AWG_FINAL} · {DEPTH_FT} ft · {I_AMPS} A · T={T_BOTTOM_C}°C")
print(f"  Caída de voltaje : {cable['V_drop_V']:.1f} V ({cable['pct_drop']:.1f}%)  ", end='')
print('✅' if not cable['warning_5pct'] else '⚠️')
print(f"  Arrhenius (H)    : ΔT={arrh['delta_T_C']:.0f}°C · vida={arrh['pct_life_remaining']:.0f}%  ", end='')
print('✅' if not arrh['warning'] else '⚠️')
print(f"  THD ({VSD_TOPO})  : {thd['THD_pct']}%  ", end='')
print('✅ IEEE 519' if thd['complies_ieee519'] else '❌ No cumple')
print(f"  Cable material   : {nace['cable_jacket']}")
print(f"  Elastómero       : {nace['elastomer_recommendation']}")

# Figura resumen
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# (0,0) AWG sensitivity
ax = axes[0, 0]
awg_arr  = [1, 2, 4, 6, 8, 10, 12, 14]
pcts_awg = [cable_voltage_drop(a, DEPTH_FT, I_AMPS, T_BOTTOM_C)['pct_drop'] for a in awg_arr]
bar_colors = ['#F472B6' if a == AWG_FINAL else ('#EF4444' if p > 10 else '#F59E0B' if p > 5 else '#22C55E')
              for a, p in zip(awg_arr, pcts_awg)]
ax.bar([f'#{a}' for a in awg_arr], pcts_awg, color=bar_colors, edgecolor='#1E293B')
ax.axhline(5,  color='#F59E0B', lw=1.2, ls='--')
ax.axhline(10, color='#EF4444', lw=1.2, ls='--')
ax.set_title(f'V_drop% por AWG (seleccionado: #{AWG_FINAL})', fontsize=10)
ax.set_ylabel('V_drop [%]')
ax.grid(True, alpha=0.4)

# (0,1) Arrhenius
ax = axes[0, 1]
T_arr = np.linspace(140, 230, 200)
ax.plot(T_arr, [arrhenius_life_factor(T, T_RATED_C)['pct_life_remaining'] for T in T_arr],
        color='#FB923C', lw=2)
ax.axvline(T_BOTTOM_C, color='#F472B6', lw=2, ls='--', label=f'T_op={T_BOTTOM_C}°C')
ax.axvline(T_RATED_C,  color='#22C55E', lw=1.5, ls=':', label=f'Límite clase H ({T_RATED_C}°C)')
ax.scatter([T_BOTTOM_C], [arrh['pct_life_remaining']], color='#F472B6', s=80, zorder=5)
ax.set_title('Curva Arrhenius — Aislamiento Clase H', fontsize=10)
ax.set_ylabel('Vida útil relativa [%]')
ax.set_xlabel('T° operación [°C]')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)
ax.set_ylim(0, 105)

# (1,0) THD
ax = axes[1, 0]
all_topos  = list(VSD_THD_TYPICAL.keys())
all_labels = [VSD_THD_TYPICAL[t]['desc'] for t in all_topos]
all_thds   = [VSD_THD_TYPICAL[t]['THD_pct'] for t in all_topos]
b_colors   = ['#F472B6' if t == VSD_TOPO else ('#22C55E' if v < IEEE_519_LIMIT_PCT else '#EF4444')
              for t, v in zip(all_topos, all_thds)]
ax.barh(all_labels, all_thds, color=b_colors, edgecolor='#1E293B')
ax.axvline(IEEE_519_LIMIT_PCT, color='#22C55E', lw=1.5, ls='--', label='IEEE 519 (5%)')
ax.set_title('THD por topología (seleccionado: 18P)', fontsize=10)
ax.set_xlabel('THD [%]')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)

# (1,1) Dashboard texto
ax = axes[1, 1]
ax.axis('off')
summary = (
    f"FLAMINGO-4 — RESUMEN ELÉCTRICO\n"
    f"{'='*35}\n\n"
    f"Cable    : AWG #{AWG_FINAL} · {DEPTH_FT} ft\n"
    f"V_drop   : {cable['pct_drop']:.1f}% ({'OK' if not cable['warning_5pct'] else 'AVISO'})\n"
    f"R_corr   : {cable['R_corrected_ohm']:.4f} Ω/conductor\n\n"
    f"T_op     : {T_BOTTOM_C}°C vs límite {T_RATED_C}°C\n"
    f"Arrhenius: ΔT={arrh['delta_T_C']:.0f}°C → vida={arrh['pct_life_remaining']:.0f}%\n\n"
    f"VSD      : 18 pulsos\n"
    f"THD      : {thd['THD_pct']}% ({'IEEE 519 OK' if thd['complies_ieee519'] else 'No cumple'})\n\n"
    f"NACE     : H₂S presente\n"
    f"Cable    : {nace['cable_jacket']}\n"
    f"Elastóm. : EPDM o PEEK\n"
)
ax.text(0.05, 0.95, summary, transform=ax.transAxes,
        fontsize=9, va='top', ha='left', color='#CBD5E1',
        fontfamily='monospace', linespacing=1.6)

plt.suptitle('Pozo Flamingo-4 — Análisis Eléctrico Integrado (post-optimización)',
             color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 6. Resumen — Reglas Operativas M4

| Parámetro | Umbral | Acción |
|-----------|--------|--------|
| V_drop | > 5% | Evaluar AWG inferior o reducir corriente |
| V_drop | > 10% | Rediseño inmediato del cable |
| ΔT aislamiento | > 10°C | Vida útil < 50% — revisar diseño térmico |
| ΔT aislamiento | > 30°C | Vida útil < 12.5% — cambiar clase de aislamiento |
| THD | > 5% | No cumple IEEE 519-2014 — requiere upgrade topología VSD |
| H₂S presente | siempre | Lead Sheath + Monel 400 obligatorios (NACE MR0175) |
| T > 140°C | siempre | NBR NO apto — usar EPDM o PEEK |

---
*Notebook generado automáticamente por SIMBES para el Módulo 4 — Eléctrico y VSD.*  
*Versión: 1.0 · Fecha: 2026-03-13*